# Ten Trails — full MF6 model

Builds every package and writes the files. Nothing is run for you: the model does not
converge past period 2, and that is the thing to look at.

Companion: `usg_import_workbook.ipynb` explores the USG side and diagnoses convergence.


In [1]:
from pathlib import Path
import warnings, logging

import numpy as np
import pandas as pd
import flopy
import myflopy as mf
from myflopy.modflow.usg import read_usg

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)-7s %(message)s')

USG_DIR = Path('/home/lukem/models/mf6/Ten Trails/SSPA/Model_FINAL_ForLuke/14-Calib40_Iter01_DV')
NAM     = USG_DIR / 'flow-tt01_USE_5yr.nam'   # 72 periods; the one d_model.bat runs
GSF     = USG_DIR / 'flow-tt01.gsf'
CRS     = 'EPSG:2927'                         # WA State Plane South, ft
START   = '2017-10-01'                        # the DISU date column is unusable, see report()
NAME    = 'tentrails'
WORK    = Path('/tmp/tentrails_mf6')

## 1 · The model


In [2]:
usg = read_usg(NAM, gsf=GSF, crs=CRS)
usg

INFO    reading MODFLOW-USG model flow-tt01_USE_5yr.nam (BAS6, SMS, DISU, OC, RCH, WEL, DRN, CHD, LPF, CLN, GHB, ETS, HFB6)
INFO    building the Voronoi grid
INFO    Voronoi grid ready: 9405 cells
WARNING flow-tt01_5yr.dis carries a date on each stress-period line, but the dates do not increase (2023-10-31 then 2023-01-31) -- so they cannot give a start date. Pass start_date_time= to to_mf6() to set one.
INFO    read 5 layers x 9405 cells, 72 periods, 4 boundary package(s), CLN with 804 nodes


UsgModel('flow-tt01_USE_5yr.nam', 5 layers x 9405 cells, 72 periods, packages=['CHD', 'DRN', 'GHB', 'WEL'])

In [3]:
# Every package. `fix_for_mf6` makes the edits MF6 requires to accept the model at all
# (GHB heads raised to their cell bottom, CHD dropped in the periods where it sits below);
# each one is logged. Drop it to see what MF6 refuses, or call usg.validate() first.
sim = usg.to_mf6(NAME, start_date_time=START, fix_for_mf6=True)
sim

WARNING CHD: omitted 348 record-period(s) whose head sits below the cell bottom -- inert in MODFLOW-USG, rejected by MODFLOW 6 (fix_for_mf6=True)
WARNING GHB: raised 6 boundary head(s) to their cell bottom so MODFLOW 6 accepts them (fix_for_mf6=True)
INFO    ETS -> list-based EVT: 72 periods x 9090 columns = 654480 records (nseg=2)
INFO    converted flow-tt01_USE_5yr.nam to MODFLOW 6: 11 packages on a 5 x 9405 DISV grid, 72 periods


Name,Summary
tentrails,gwf; packages=11
Name,Summary
tdis,flopy.ModflowTdis; 4 options
ims,build_ims; 16 options


## 2 · Write it

~20 s. The segmented EVT is list-based and 61 MB — that is the physics, not a format choice.


In [4]:
import shutil
shutil.rmtree(WORK, ignore_errors=True)

project = mf.Project(WORK, name=NAME)
project.add_simulation(sim)
run = project.prepare_run('full', sim)
ws = run.write()          # writes, does NOT run

print(ws)
print(f'{sum(f.stat().st_size for f in ws.iterdir()) / 1e6:.0f} MB, {len(list(ws.iterdir()))} files')
print(f'\nrun it:  cd {ws} && mf6')

/tmp/tentrails_mf6/runs/full
80 MB, 16 files

run it:  cd /tmp/tentrails_mf6/runs/full && mf6


In [22]:
run.model().gwf.oc

package_name = oc
filename = tentrails.oc
package_type = oc
model_or_simulation_package = model
model_name = tentrails

Block options
--------------------
budget_filerecord
{internal}
(rec.array([('tentrails.cbc',)],
          dtype=[('budgetfile', 'O')]))

head_filerecord
{internal}
(rec.array([('tentrails.hds',)],
          dtype=[('headfile', 'O')]))


Block period
--------------------
saverecord
{internal}
(rec.array([('HEAD', 'ALL', None), ('BUDGET', 'ALL', None)],
          dtype=[('rtype', 'O'), ('ocsetting', 'O'), ('ocsetting_data', 'O')]))

printrecord
None


## 3 · Run it yourself

The cell above prints the command. A terminal is better than a notebook here —
you get the outer-iteration table as it scrolls.

Or from here, with live output:


In [ ]:
success, report = run.execute(silent=False)
print(success)

## 4 · Afterwards


In [ ]:
model = run.model(NAME)
model.file_summary()

In [ ]:
# where it stopped
hds = flopy.utils.HeadFile(ws / f'{NAME}.hds')
kk = hds.get_kstpkper()
print(f'{len(kk)} records, last = period {kk[-1][1] + 1} of {usg.nper}')